# Imports

In [1]:
import sys
sys.path.append("/kaggle/input/datasets/yusufhilal/full-model/Event-Extraction-Model")

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import json
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path
from functools import partial
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from helpers.dataset import PageDataset, combine_pages
from helpers.metrics import pick_starts_from_probs
from models.dom_extractor import DOMAwareEventExtractor
from models.field_classifier import build_features

from inference import load_models, load_data_and_prepare, run_dom_extractor, run_field_classifier, predict_events, save_output

# Loading the Models and Preprocessing

In [4]:
# loading the models -----------------------------------------------
model, tokenizer, ckpt, field_bundle = load_models(
    checkpoint_path="/kaggle/input/datasets/yusufhilal/models/dom_extractor_checkpoint_v2.pt",
    classifier_path="/kaggle/input/datasets/yusufhilal/models/field_classifier_v1.joblib",
    device=device
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
print("Model loaded successfully")
print(f"Device: {device}")
print(f"Best threshold: {ckpt['best_th']}")
print(f"Num cols: {ckpt['num_cols']}")
print(f"Bool cols: {ckpt['bool_cols']}")
print(f"Field classes: {list(field_bundle['label_encoder'].classes_)}")

Model loaded successfully
Device: cuda
Best threshold: 0.12379999999999991
Num cols: ['depth', 'sibling_index', 'children_count', 'same_tag_sibling_count', 'same_text_sibling_count', 'text_length', 'word_count', 'letter_ratio', 'digit_ratio', 'whitespace_ratio', 'attribute_count']
Bool cols: ['has_link', 'link_is_absolute', 'parent_has_link', 'is_leaf', 'contains_date', 'contains_time', 'starts_with_digit', 'ends_with_digit', 'has_class', 'has_id', 'attr_has_word_name', 'attr_has_word_date', 'attr_has_word_time', 'attr_has_word_location', 'attr_has_word_link', 'text_has_word_name', 'text_has_word_date', 'text_word_time', 'text_word_description', 'text_word_location']
Field classes: ['Date', 'Description', 'Location', 'Name', 'Price', 'Time']


In [6]:
df = load_data_and_prepare(
    csv_path="/kaggle/input/datasets/yusufhilal/test-data/test_data.csv",
    ckpt=ckpt,
)

print(f"Sources: {df['source'].unique()}")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Sources: ['neacac_spring.net_pattern_labeled' 'pnacac_spring.org_pattern_labeled']
Shape: (274, 39)
Columns: ['rendering_order', 'tag', 'attributes', 'text_context', 'depth', 'parent_index', 'parent_tag', 'text_length', 'sibling_index', 'link', 'children_count', 'same_tag_sibling_count', 'same_text_sibling_count', 'has_link', 'link_is_absolute', 'parent_has_link', 'is_leaf', 'word_count', 'letter_ratio', 'digit_ratio', 'whitespace_ratio', 'contains_date', 'contains_time', 'starts_with_digit', 'ends_with_digit', 'attribute_count', 'has_class', 'has_id', 'attr_has_word_name', 'attr_has_word_date', 'attr_has_word_time', 'attr_has_word_location', 'attr_has_word_link', 'text_has_word_name', 'text_has_word_date', 'text_word_time', 'text_word_description', 'text_word_location', 'source']


In [7]:
results = run_dom_extractor(df, model, tokenizer, ckpt, device)

bio_map = {0: "O", 1: "B", 2: "I"}

for result in results:
    if not result:
        continue
    
    source = result["source"]
    probs_full_valid = result["probs_full_valid"]
    page = df[df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
    
    pred_bio = probs_full_valid.argmax(axis=1)
    prob_B = probs_full_valid[:, 1]
    
    print(f"\n{'='*80}")
    print(f"Source: {source}")
    print(f"{'='*80}")
    
    for node_idx in range(len(page)):
        bio = bio_map[pred_bio[node_idx]]
        prob_b = f"{prob_B[node_idx]:.3f}"
        text = str(page.iloc[node_idx]["text_context"])
        print(f"{node_idx} | BIO={bio} | ProbB={prob_b} | {text}")


Source: neacac_spring.net_pattern_labeled
0 | BIO=O | ProbB=0.002 | search
1 | BIO=O | ProbB=0.002 | member login
2 | BIO=O | ProbB=0.002 | home
3 | BIO=O | ProbB=0.002 | about
4 | BIO=O | ProbB=0.002 | about neacac
5 | BIO=O | ProbB=0.002 | mission statement
6 | BIO=O | ProbB=0.002 | strategic framework
7 | BIO=O | ProbB=0.002 | contact us
8 | BIO=O | ProbB=0.002 | job board
9 | BIO=O | ProbB=0.002 | governance
10 | BIO=O | ProbB=0.002 | executive board
11 | BIO=O | ProbB=0.002 | leadership & committee reports
12 | BIO=O | ProbB=0.002 | neacac bylaws and articles of organization
13 | BIO=O | ProbB=0.002 | committees
14 | BIO=O | ProbB=0.002 | about committees
15 | BIO=O | ProbB=0.002 | admission practices
16 | BIO=O | ProbB=0.006 | college fair
17 | BIO=O | ProbB=0.003 | annual meeting & conference
18 | BIO=O | ProbB=0.002 | articles of organization & by-laws
19 | BIO=O | ProbB=0.002 | communication and web services
20 | BIO=O | ProbB=0.002 | finance
21 | BIO=O | ProbB=0.002 | founda

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


In [9]:
results = run_dom_extractor(df, model, tokenizer, ckpt, device)
sources = list(df.groupby("source", sort=False).groups.keys())

print(f"Pages processed: {len(results)}")
for i, result in enumerate(results):
    if result is None:
        print(f"Page {i}: no events found")
        continue
    
    source = result["source"]
    start_indices = result["start_indices"]
    page = df[df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
    
    print(f"\nPage {i} ({source}): {len(start_indices)} events found")
    for page_idx in start_indices:
        print(f"  start node {page_idx}: {page.iloc[page_idx]['text_context']}")

Pages processed: 2

Page 0 (neacac_spring.net_pattern_labeled): 8 events found
  start node 149: wednesday, march 25th
  start node 156: thursday, april 23rd
  start node 161: merrimack college
  start node 167: university of maine augusta
  start node 173: assumption university
  start node 179: bridgewater state university
  start node 185: franklin pierce university
  start node 189: grimshaw-gudiewicz northfields activity center "the bubble"

Page 1 (pnacac_spring.org_pattern_labeled): 4 events found
  start node 11: saturday, april 1
  start node 19: thursday, april 2
  start node 26: saturday, april 2
  start node 33: sunday, april 2


In [10]:
for result in results:
    if not result:
        continue
    
    source = result["source"]
    valid = result["valid"]
    probs_full_valid = result["probs_full_valid"]
    page = df[df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
    
    print(f"Source: {source}")
    print(f"Page nodes: {len(page)}")
    print(f"Valid nodes: {len(valid)}")
    print(f"probs_full_valid shape: {probs_full_valid.shape}")
    print(f"valid: {valid}")

Source: neacac_spring.net_pattern_labeled
Page nodes: 233
Valid nodes: 233
probs_full_valid shape: (233, 3)
valid: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 

# Field Classifier

In [11]:
node_labels = run_field_classifier(df, results, field_bundle)

for i, (result, labels) in enumerate(zip(results, node_labels)):
    if result is None or labels is None:
        print(f"Page {i}: skipped")
        continue
    
    source = result["source"]
    probs_full_valid = result["probs_full_valid"]
    page = df[df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
    
    bio_map = {0: "O", 1: "B", 2: "I"}
    pred_bio = probs_full_valid.argmax(axis=1)
    prob_B = probs_full_valid[:, 1]
    
    print(f"\n{'='*100}")
    print(f"Page {i} ({source})")
    print(f"{'='*100}")
    print(f"{'Idx':<6} {'RenderOrd':<12} {'BIO_true':<10} {'BIO_pred':<10} {'ProbB':<8} {'Field':<12} {'Text'}")
    print(f"{'-'*100}")
    
    for node_idx in range(len(page)):
        row = page.iloc[node_idx]
        bio_true = row.get("bio", "?")
        bio_pred = bio_map[pred_bio[node_idx]]
        prob_b = f"{prob_B[node_idx]:.3f}"
        field = labels.get(node_idx, "-")
        text = str(row["text_context"])[:40]
        render = row["rendering_order"]
        
        print(f"{node_idx:<6} {render:<12} {str(bio_true):<10} {bio_pred:<10} {prob_b:<8} {field:<12} {text}")


Page 0 (neacac_spring.net_pattern_labeled)
Idx    RenderOrd    BIO_true   BIO_pred   ProbB    Field        Text
----------------------------------------------------------------------------------------------------
0      36           ?          O          0.002    -            search
1      41           ?          O          0.002    -            member login
2      46           ?          O          0.002    -            home
3      49           ?          O          0.002    -            about
4      53           ?          O          0.002    -            about neacac
5      56           ?          O          0.002    -            mission statement
6      59           ?          O          0.002    -            strategic framework
7      62           ?          O          0.002    -            contact us
8      65           ?          O          0.002    -            job board
9      68           ?          O          0.002    -            governance
10     72           ?          O

# Event Prediction and Restructuring

In [12]:
events = predict_events(df, results, node_labels)

for event in events:
    print(event)

{'source': 'neacac_spring.net_pattern_labeled', 'event_number': 1, 'Name': 'bishop guertin high school', 'Date': 'wednesday, march 25th', 'Time': '6pm', 'Time_2': 'open to the public at 6:30pm', 'Time_3': '8pm', 'Location': 'roedell field house', 'Location_2': 'nashua, nh'}
{'source': 'neacac_spring.net_pattern_labeled', 'event_number': 2, 'Name': 'mitchell college', 'Date': 'thursday, april 23rd', 'Time': '9am', 'Time_2': '11am', 'Location': 'yarnall athletic center', 'Location_2': 'new london, ct'}
{'source': 'neacac_spring.net_pattern_labeled', 'event_number': 3, 'Name': 'merrimack college', 'Date': 'sunday, may 3rd', 'Time': '6pm', 'Time_2': '8pm', 'Location': 'merrimack athletic complex: lawler rink', 'Location_2': 'north andover, ma'}
{'source': 'neacac_spring.net_pattern_labeled', 'event_number': 4, 'Name': 'university of maine augusta', 'Date': 'thursday, may 14th', 'Time': '9am', 'Time_2': '11am', 'Location': 'augusta civic center', 'Location_2': 'augusta, me'}
{'source': 'nea

In [13]:
from collections import defaultdict
page_events = defaultdict(list)

for event in events:
    page_events[event["source"]].append(event)

for source, evs in page_events.items():
    print(f"\nSource: {source} | {len(evs)} events")
    for ev in evs:
        # count fields excluding source and event_number
        num_fields = len([k for k in ev.keys() if k not in ("source", "event_number")])
        print(f"  Event {ev['event_number']}: {num_fields} fields — {list(ev.keys())}")


Source: neacac_spring.net_pattern_labeled | 7 events
  Event 1: 7 fields — ['source', 'event_number', 'Name', 'Date', 'Time', 'Time_2', 'Time_3', 'Location', 'Location_2']
  Event 2: 6 fields — ['source', 'event_number', 'Name', 'Date', 'Time', 'Time_2', 'Location', 'Location_2']
  Event 3: 6 fields — ['source', 'event_number', 'Name', 'Date', 'Time', 'Time_2', 'Location', 'Location_2']
  Event 4: 6 fields — ['source', 'event_number', 'Name', 'Date', 'Time', 'Time_2', 'Location', 'Location_2']
  Event 5: 6 fields — ['source', 'event_number', 'Name', 'Date', 'Time', 'Time_2', 'Location', 'Location_2']
  Event 6: 6 fields — ['source', 'event_number', 'Name', 'Date', 'Time', 'Time_2', 'Location', 'Location_2']
  Event 7: 6 fields — ['source', 'event_number', 'Name', 'Date', 'Time', 'Time_2', 'Location', 'Location_2']

Source: pnacac_spring.org_pattern_labeled | 4 events
  Event 1: 8 fields — ['source', 'event_number', 'Name', 'Date', 'Date_Location_1', 'Date_Location_2', 'Date_Location_3

# Saving Output

In [20]:
# save
save_output(events, "sample_output.json")